# 📈 Learning Rate Schedules: Cosine vs. WSD (Warmup-Stable-Decay)

Welcome! In this notebook, we explore <b>Learning Rate Schedulers</b>—the throttle control of foundation model pretraining.

---

## 1. The Metaphor: The Spacecraft Landing 🚀

Imagine piloting a spacecraft landing on a new planet:
<ol>
  <li><b>Warmup Phase (Takeoff):</b> Gently ramp up throttle from 0% to 100% so the ship doesn't shake apart from cold engines (prevents gradient explosion).</li>
  <li><b>Stable Phase (Cruising):</b> Cruise at maximum constant speed across the vast distance to learn general representations efficiently.</li>
  <li><b>Decay Phase (Landing):</b> As you approach the surface, throttle down by 90% to make a pinpoint landing into the exact optimal valley.</li>
</ol>

---
## 2. Visual Architecture: Cosine vs. WSD (Warmup-Stable-Decay)

<div style='display: flex; justify-content: center; margin: 20px 0;'>
<svg width='820' height='290' viewBox='0 0 820 290' xmlns='http://www.w3.org/2000/svg' style='background: #0d1117; border: 1px solid #30363d; border-radius: 12px; font-family: -apple-system, sans-serif;'>
  <!-- Panel 1: Cosine -->
  <g transform='translate(30, 25)'>
    <rect width='360' height='240' rx='10' fill='#161b22' stroke='#30363d' stroke-width='1.5'/>
    <text x='20' y='32' fill='#d29922' font-size='15' font-weight='bold'>1. Classic Cosine Decay</text>
    <text x='20' y='52' fill='#8b949e' font-size='12'>Used in GPT-3 and LLaMA 1 / 2</text>
    
    <!-- Axes -->
    <line x1='50' y1='180' x2='320' y2='180' stroke='#484f58' stroke-width='1.5'/>
    <line x1='50' y1='70' x2='50' y2='180' stroke='#484f58' stroke-width='1.5'/>
    <text x='42' y='75' fill='#8b949e' font-size='10' text-anchor='end'>LR</text>
    <text x='320' y='198' fill='#8b949e' font-size='10' text-anchor='end'>Steps (T)</text>
    
    <!-- Curve: Warmup then continuous cosine decay -->
    <path d='M 50 180 L 80 85 C 130 85, 230 150, 310 172' fill='none' stroke='#d29922' stroke-width='3.5' stroke-linecap='round'/>
    
    <!-- Annotations -->
    <text x='80' y='210' fill='#8b949e' font-size='11' text-anchor='middle'>Warmup</text>
    <line x1='80' y1='180' x2='80' y2='195' stroke='#8b949e' stroke-width='1' stroke-dasharray='2,2'/>
    <text x='200' y='110' fill='#d29922' font-size='11' font-weight='600'>Continuous Decay</text>
    <text x='20' y='230' fill='#f85149' font-size='11'>❌ Fixed budget: Cannot cleanly extend run!</text>
  </g>

  <!-- Panel 2: WSD -->
  <g transform='translate(430, 25)'>
    <rect width='360' height='240' rx='10' fill='#161b22' stroke='#30363d' stroke-width='1.5'/>
    <text x='20' y='32' fill='#58a6ff' font-size='15' font-weight='bold'>2. Modern WSD (Warmup-Stable-Decay)</text>
    <text x='20' y='52' fill='#8b949e' font-size='12'>Used in MiniCPM and Mirendil-Base</text>
    
    <!-- Axes -->
    <line x1='50' y1='180' x2='320' y2='180' stroke='#484f58' stroke-width='1.5'/>
    <line x1='50' y1='70' x2='50' y2='180' stroke='#484f58' stroke-width='1.5'/>
    <text x='42' y='75' fill='#8b949e' font-size='10' text-anchor='end'>LR</text>
    <text x='320' y='198' fill='#8b949e' font-size='10' text-anchor='end'>Steps</text>
    
    <!-- Curve: Warmup, Stable Plateau, Steep Decay -->
    <path d='M 50 180 L 80 85 L 245 85 C 275 85, 295 160, 310 172' fill='none' stroke='#58a6ff' stroke-width='3.5' stroke-linecap='round'/>
    
    <!-- Shaded Plateau -->
    <rect x='80' y='85' width='165' height='95' fill='#58a6ff' fill-opacity='0.08'/>
    <text x='162' y='130' fill='#58a6ff' font-size='12' font-weight='bold' text-anchor='middle'>STABLE PLATEAU</text>
    <text x='162' y='146' fill='#8b949e' font-size='10' text-anchor='middle'>(Train as long as you want!)</text>
    
    <text x='280' y='125' fill='#3fb950' font-size='11' font-weight='bold'>Decay</text>
    <text x='20' y='230' fill='#3fb950' font-size='11'>✅ Flexible: Trigger decay whenever ready!</text>
  </g>
</svg>
</div>

---
## 3. Exercise 1: Pencil & Paper Calculation 📝

Suppose our training run has:
- Max Learning Rate = <b>0.001</b>
- Total Steps = <b>100</b>
- Warmup = Steps 0 to 20
- Stable = Steps 20 to 80
- Decay = Steps 80 to 100 (anneals down to 0.0001)

### Calculate the LR at 4 Key Points on Paper:
1. <b>Step 10 (Halfway through warmup):</b><br>
   LR = 0.001 · (10 / 20) = <b>0.0005</b>
<br><br>
2. <b>Step 50 (Middle of Stable phase):</b><br>
   LR = <b>0.0010</b> (Full throttle!)
<br><br>
3. <b>Step 80 (End of Stable phase):</b><br>
   LR = <b>0.0010</b> (Still full throttle!)
<br><br>
4. <b>Step 100 (Final landing step):</b><br>
   LR = <b>0.0001</b> (90% decay &rarr; settles into sharp global minimum!)

<br>
<i>Notice: During steps 80 to 100, validation loss plunges like a waterfall!</i>

---
## 4. Visual Comparison: Cosine vs. WSD in Matplotlib

Run this cell to plot our actual project schedulers side-by-side:

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
import matplotlib.pyplot as plt
from src.engine.optim import get_lr_scheduler

total_steps = 1000
warmup_steps = 100

# Mock optimizer
p = torch.nn.Parameter(torch.zeros(1))
opt_cosine = torch.optim.SGD([p], lr=1e-3)
opt_wsd = torch.optim.SGD([p], lr=1e-3)

sched_cosine = get_lr_scheduler(opt_cosine, scheduler_type='cosine', warmup_steps=warmup_steps, total_steps=total_steps)
sched_wsd = get_lr_scheduler(opt_wsd, scheduler_type='wsd', warmup_steps=warmup_steps, total_steps=total_steps, stable_ratio=0.8)

lrs_cosine = []
lrs_wsd = []

for step in range(total_steps):
    lrs_cosine.append(sched_cosine.get_last_lr()[0])
    lrs_wsd.append(sched_wsd.get_last_lr()[0])
    sched_cosine.step()
    sched_wsd.step()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(lrs_cosine, label='Cosine Schedule (Decays continuously)', color='#d29922', lw=2)
ax.plot(lrs_wsd, label='WSD Schedule (Stable plateau then decay)', color='#58a6ff', lw=2.5)

ax.set_title("Learning Rate Schedule: Cosine vs. WSD (Warmup-Stable-Decay)", fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel("Training Steps", fontsize=10)
ax.set_ylabel("Learning Rate", fontsize=10)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()
print("SUCCESS: Schedulers traced and plotted cleanly!")